# 01 · Datasets: built-in samples and your own data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/01_datasets.ipynb)

**Runtime:** CPU is fine. This notebook only samples data; no models run.

← [00_start_here](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/00_start_here.ipynb) · [series index](https://github.com/jstjoe/local-privacy/blob/main/notebooks/README.md) · [02_run_detectors](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/02_run_detectors.ipynb) →

A benchmark is only as good as its gold labels. This notebook covers:

- how to pick a deterministic **sample** from a built-in dataset
- what a **fixture record** looks like, and how to check a fixtures file
- how to bring **your own data**: already-labelled JSONL, CSV with your own
  label names, or token-tagged (BIO) data

Your choices are saved as the **session**, so the following notebooks
benchmark the same sample.

In [ ]:
# Setup — run once per session. On Colab this clones and installs the harness
# (a few minutes the first time). Locally, after `uv sync`, it does nothing.
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_REF = "main"   # branch or tag — e.g. a PR branch to try it before merging
OPF_REPO = "https://github.com/openai/privacy-filter.git"
OPF_REF = "main"
USE_DRIVE = False      # True: keep results in Google Drive so the other notebooks reuse them

import importlib.util, os, subprocess, sys

if importlib.util.find_spec("opf_eval") is None:
    home = "/content" if "google.colab" in sys.modules else os.path.expanduser("~")
    for repo, ref, dest in [(OPF_REPO, OPF_REF, f"{home}/privacy-filter"),
                            (HARNESS_REPO, HARNESS_REF, f"{home}/local-privacy")]:
        if not os.path.exists(dest):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", ref, repo, dest], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{home}/privacy-filter",
                    f"{home}/local-privacy/eval[notebooks]", f"{home}/local-privacy/api"], check=True)
    # A running kernel doesn't always see freshly pip-installed packages;
    # putting the sources on sys.path makes the imports below work either way.
    sys.path[:0] = [f"{home}/privacy-filter", f"{home}/local-privacy/eval/src", f"{home}/local-privacy/api/src"]

from opf_eval import nb
ws = nb.setup(drive=USE_DRIVE)

## Choose a sample

A sample is `(dataset, n, seed)`. The same three values always give the same
records, so results are reproducible and re-runs reuse the file on disk.

- `pii_masking_200k` is the default. It annotates all 15 coarse categories,
  the richest of the built-ins.
- `openpii_nano` (1k records) is the quickest to download.
- `n=100` is a smoke test, `n=1000` a real benchmark, and `n=5000` gives
  stable per-category numbers.

In [ ]:
# === EDIT ME ===
S = nb.session(
    dataset="pii_masking_200k",   # see datasets.describe() in 00, or your own (below)
    n=100,                        # None = every record (sensible for small custom files)
    seed=42,
)
# ================
S.show()

## Materialize the fixtures

*Materializing* downloads the dataset (cached after the first time), draws the
sample, and maps every gold label into the canonical taxonomy. The result is a
**fixtures file**: one JSON record per line, plus a `.meta.json` sidecar that
records where the sample came from and which labels it annotates.

If a file for exactly this sample already exists and is unchanged, it's
reused. A different seed or size always gets its own file.

In [ ]:
from opf_eval import fixtures
from opf_eval.io import read_jsonl, read_meta

fx_path = nb.ensure_fixtures(S)
records = read_jsonl(fx_path)
meta = read_meta(fx_path)
print(f"{meta['n_written']} records from {meta['source']}")
print(f"annotated labels ({len(meta['labels'])}): {', '.join(meta['labels'])}")

## Anatomy of a fixture record

```json
{"id": "…", "text": "…", "language": "en",
 "gold_spans": [{"label": "PERSON", "fine_label": "GIVEN_NAME",
                 "raw_label": "FIRSTNAME", "start": 17, "end": 22}]}
```

`start` and `end` are character offsets into `text`, end-exclusive (Python
slicing). `label` is coarse, `fine_label` is the most specific canonical
label, and `raw_label` is the dataset's own label.

In [ ]:
rec = records[0]
print(rec["text"], "\n")
nb.show_table(
    [{**s, "text": rec["text"][s["start"]:s["end"]]} for s in rec["gold_spans"]],
    ["text", "label", "fine_label", "raw_label", "start", "end"],
)

## Check the file

`validate_fixtures` catches problems that would otherwise silently skew
scores:

- **Errors** break the contract: duplicate ids, or offsets outside the text.
- **Warnings** flag things scoring will ignore: labels outside the taxonomy,
  or overlapping gold spans.

It also shows the label distribution. A category with only a handful of
examples will have a noisy F1.

In [ ]:
report = fixtures.validate_fixtures(fx_path)
print(report.summary())

## Bring your own dataset

Anything with text and character-offset spans works. Register it once with
`datasets.register_dataset(...)`, then use its name anywhere a built-in
dataset name works.

### 1 · Already in canonical labels

The simplest case: JSONL with a `text` field and `spans` of
`{start, end, label}`, using canonical label names (see the table in 00).
`n=None` keeps every record, in file order.

In [ ]:
import json
from opf_eval import datasets

tickets = [
    {"id": "t1", "text": "Customer Maria Lopez (maria.lopez@example.com) can't log in.",
     "spans": [{"start": 9, "end": 14, "label": "GIVEN_NAME"},
               {"start": 15, "end": 20, "label": "FAMILY_NAME"},
               {"start": 22, "end": 45, "label": "EMAIL"}]},
    {"id": "t2", "text": "Refund to IBAN DE89 3704 0044 0532 0130 00 please.",
     "spans": [{"start": 15, "end": 42, "label": "BANK_ACCOUNT"}]},
    {"id": "t3", "text": "Employee E-20931 reset the router.",
     "spans": [{"start": 9, "end": 16, "label": "EMPLOYEE_ID"}]},
]
my_file = ws.data / "my_tickets.jsonl"
my_file.write_text("".join(json.dumps(r) + "\n" for r in tickets))

datasets.register_dataset("my_tickets", path=my_file, description="hand-labelled support tickets")
path, _ = fixtures.ensure_fixtures(ws.data / "my_tickets_all.jsonl", None, dataset="my_tickets")
print(fixtures.validate_fixtures(path).summary())

Some fields also accept common alternative names: `text` can be `content`,
`spans` can be `entities`, and `start`/`end` can be `start_offset`/`end_offset`,
among others. For anything else, name the fields explicitly:
`register_dataset(..., text_field="body", spans_field="pii", label_key="kind")`.

### 2 · Your own label names (CSV)

Map your labels onto the taxonomy with `label_map`. Any label you leave out is
dropped. CSV can have one row per record (with a JSON `spans` column) or, as
here, one row per span.

In [ ]:
import csv

rows = [
    {"id": "c1", "text": "Patient: J. Okafor, MRN 00482913, DOB 1984-02-11", "label": "patient_name", "start": 9, "end": 18},
    {"id": "c1", "text": "Patient: J. Okafor, MRN 00482913, DOB 1984-02-11", "label": "mrn", "start": 24, "end": 32},
    {"id": "c1", "text": "Patient: J. Okafor, MRN 00482913, DOB 1984-02-11", "label": "dob", "start": 38, "end": 48},
    {"id": "c2", "text": "Follow-up booked, no identifiers.", "label": "", "start": "", "end": ""},
]
claims_file = ws.data / "claims.csv"
with claims_file.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["id", "text", "label", "start", "end"])
    w.writeheader()
    w.writerows(rows)

datasets.register_dataset(
    "claims",
    path=claims_file,
    label_map={"patient_name": "PERSON", "mrn": "MEDICAL_ID", "dob": "DATE_OF_BIRTH"},
)
path, _ = fixtures.ensure_fixtures(ws.data / "claims_all.jsonl", None, dataset="claims")
for r in read_jsonl(path):
    print(r["id"], [(r["text"][s["start"]:s["end"]], s["fine_label"]) for s in r["gold_spans"]])

Declare `labels=[...]` when your label map covers more categories than you
actually annotated. Detectors are only charged for predicting categories the
dataset claims to annotate.

### 3 · Token-tagged (BIO) data

CoNLL-style datasets give tokens and `B-`/`I-` tags instead of character
offsets. `schema="bio"` rebuilds the text and the offsets. BIOES/BILOU tags
and integer tag ids (`tag_names=[...]`) work too.

In [ ]:
bio_file = ws.data / "bio_sample.jsonl"
bio_file.write_text(json.dumps({
    "tokens": ["Contact", "Ana", "Silva", "at", "ana@example.pt", "."],
    "ner_tags": ["O", "B-PER", "I-PER", "O", "B-MAIL", "O"],
    "trailing_whitespace": [True, True, True, True, False, False],
}) + "\n")

datasets.register_dataset("bio_sample", path=bio_file, schema="bio",
                          label_map={"PER": "PERSON", "MAIL": "EMAIL"})
path, _ = fixtures.ensure_fixtures(ws.data / "bio_sample_all.jsonl", None, dataset="bio_sample")
(r,) = read_jsonl(path)
print(repr(r["text"]), [(r["text"][s["start"]:s["end"]], s["label"]) for s in r["gold_spans"]])

### Benchmark your dataset in the rest of the series

Point the session at it. Notebooks 02–06 reuse the fixtures file from the
sidecar, so they don't need the `register_dataset` call again.

```python
S = nb.session(dataset="my_tickets", n=None)
nb.ensure_fixtures(S)
```

From the command line, the same thing works without any Python:

```sh
python -m opf_eval.fixtures --file my_tickets.jsonl --out eval/data/mine.jsonl
python -m opf_eval.fixtures validate eval/data/mine.jsonl
```

### Public benchmarks

Recent public PII benchmarks use the same span format and will be registered
the same way. They include Tonic's PrivacyBench (human-annotated split),
NVIDIA Nemotron-PII and the Text Anonymization Benchmark (TAB). See
[`plans/12-roadmap.md`](https://github.com/jstjoe/local-privacy/blob/main/plans/12-roadmap.md).

## Next

Run the detectors over this sample.

**[02_run_detectors](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/02_run_detectors.ipynb)** — Run detectors over a sample.